# Q11.
```{admonition}
:class: note
We will now try to predict per capita crime rate in the `Boston` data set.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, KFold, cross_val_score, cross_validate
import matplotlib.pyplot as plt
from mlxtend.feature_selection import ExhaustiveFeatureSelector as EFS

In [2]:
from sklearn.linear_model import LinearRegression, LassoCV, RidgeCV, Lasso, Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.cross_decomposition import PLSRegression
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
boston = pd.read_csv('../../../ALL CSV FILES - 2nd Edition/Boston.csv',index_col=0)

In [4]:
boston.sample(3)

,crim,zn,indus,chas,nox,rm,age,dis,rad,tax,ptratio,lstat,medv
57,0.02055,85.0,0.74,0,0.410,6.383,35.7,9.1876,2,313,17.3,5.77,24.7
69,0.13554,12.5,6.07,0,0.409,5.594,36.8,6.4980,4,345,18.9,13.09,17.4
272,0.16211,20.0,6.96,0,0.464,6.240,16.3,4.4290,3,223,18.6,6.59,25.2


## (a)
```{admonition}
:class: note
Try out some of the regression methods explored in this chapter, such as best subset selection, the lasso, ridge regression, and PCR. 

In [87]:
X = boston.drop(columns=['crim'])
Y = boston['crim']

In [88]:
X_t, X_ho, Y_t, Y_ho = train_test_split(X,Y,random_state=314,shuffle=True)

In [106]:
outer_cv = KFold(shuffle=True, random_state=271)
outer_scores_subset = []

for train_idx, test_idx in outer_cv.split(X_t,Y_t):
    X_train, Y_train = X_t.iloc[train_idx], Y_t.iloc[train_idx]
    X_test, Y_test = X_t.iloc[test_idx], Y_t.iloc[test_idx]

    efs = EFS(LinearRegression(),min_features=1,max_features=X.shape[1],scoring='r2',cv=5,print_progress=False,n_jobs=-1)
    efs.fit(X_train,Y_train)

    lr = LinearRegression()
    best_features = list(efs.best_feature_names_)
    lr.fit(X_train[best_features],Y_train)
    Y_pred = lr.predict(X_test[best_features])

    outer_mse_subset = mean_squared_error(Y_test,Y_pred)
    outer_scores_subset.append((outer_mse_subset,best_features))
    

subset_cv_mse = np.mean([m for m,*_ in outer_scores_subset])
best_mse_subset, best_features_subset = min(outer_scores_subset, key=lambda x: x[0])
best_subset_lr = LinearRegression()
best_subset_lr.fit(X_t[best_features_subset],Y_t)
best_subset_lr_preds = best_subset_lr.predict(X_ho[best_features_subset])
best_subset_lr_mse = mean_squared_error(Y_ho,best_subset_lr_preds)
best_subset_lr_r2 = r2_score(Y_ho,best_subset_lr_preds)

print(f'CV mean MSE: {subset_cv_mse:.4f}')
print(f'Holdout MSE: {best_subset_lr_mse:.4f}')
print(f'Holdout R2: {best_subset_lr_r2:.4f}')
print(f'Features used: {best_features_subset}')

CV mean MSE: 41.9530
Holdout MSE: 46.7562
Holdout R2: 0.4377
Features used: ['zn', 'nox', 'dis', 'rad', 'lstat']


In [108]:
pipe_lasso = Pipeline(
    [
        ('scale',StandardScaler()),
        ('lasso',LassoCV(alphas=np.logspace(-4,4,100),cv=5))
    ]
)

cv_results_lasso = cross_validate(
    pipe_lasso, X_t, Y_t,cv=5,scoring='neg_mean_squared_error',return_estimator=True
)
cv_lasso_mse = (-cv_results_lasso['test_score']).mean()

best_lasso_alpha = [e.named_steps['lasso'].alpha_ for e in cv_results_lasso['estimator']][np.argmax(cv_results_lasso['test_score'])]

best_lasso = Pipeline(
    [
        ('scale',StandardScaler()),
        ('lasso',Lasso(alpha=best_lasso_alpha))
    ]
)

best_lasso.fit(X_t,Y_t)
best_lasso_preds = best_lasso.predict(X_ho)
best_lasso_mse = mean_squared_error(Y_ho,best_lasso_preds)
best_lasso_r2 = r2_score(Y_ho,best_lasso_preds)

print(f'CV mean MSE: {cv_lasso_mse:.4f}')
print(f'Holdout MSE: {best_lasso_mse:.4f}')
print(f'Holdout R2: {best_lasso_r2:.4f}')
print(f'Lambda used: {best_lasso_alpha}')

CV mean MSE: 41.5721
Holdout MSE: 48.6508
Holdout R2: 0.4149
Lambda used: 0.3593813663804629


In [91]:
pipe_ridge = Pipeline(
    [
        ('scale',StandardScaler()),
        ('ridge',RidgeCV(alphas=np.logspace(-4,4,100),cv=5))
    ]
)

cv_results_ridge = cross_validate(
    pipe_ridge, X_t, Y_t,cv=5,scoring='neg_mean_squared_error',return_estimator=True
)
cv_ridge_mse = np.mean(-cv_results_ridge['test_score'])

best_ridge_alpha = [e.named_steps['ridge'].alpha_ for e in cv_results_ridge['estimator']][np.argmax(cv_results_ridge['test_score'])]

best_ridge = Pipeline(
    [
        ('scale',StandardScaler()),
        ('ridge',Ridge(alpha=best_ridge_alpha))
    ]
)

best_ridge.fit(X_t,Y_t)
best_ridge_preds = best_ridge.predict(X_ho)
best_ridge_mse = mean_squared_error(Y_ho,best_ridge_preds)
best_ridge_r2 = r2_score(Y_ho,best_ridge_preds)

print(f'CV average MSE: {cv_ridge_mse:.4f}')
print(f'Holdout MSE: {best_ridge_mse:.4f}')
print(f'Holdout R2: {best_ridge_r2:.4f}')
print(f'Lambda used: {best_ridge_alpha}')

CV average MSE: 41.7941
Holdout MSE: 48.2294
Holdout R2: 0.4200
Lambda used: 79.24828983539186


In [117]:
pipe_pcr = Pipeline(
    [
        ('scale',StandardScaler()),
        ('pca',PCA()),
        ('regression',LinearRegression())
    ]
)

inner_cv = KFold(n_splits=5,shuffle=True,random_state=1728)
outer_cv = KFold(n_splits=5,shuffle=True,random_state=1729)

gridsearch = GridSearchCV(pipe_pcr,{'pca__n_components':range(1,X_t.shape[1]+1)},scoring='neg_mean_squared_error',cv=inner_cv)
cv_pcr_mse = -cross_val_score(gridsearch,X_t,Y_t,cv=outer_cv).mean()

gridsearch.fit(X_t,Y_t)
best_pcr_components = gridsearch.best_params_['pca__n_components']
best_pcr_preds = gridsearch.predict(X_ho)
best_pcr_mse= mean_squared_error(Y_ho,best_pcr_preds)
best_pcr_r2 = r2_score(Y_ho,best_pcr_preds)

print(f'CV average MSE: {cv_pcr_mse:.4f}')
print(f'Holdout MSE: {best_pcr_mse:.4f}')
print(f'Holdout R2: {best_pcr_r2:.4f}')
print(f'Components used: {best_pcr_components}')

CV average MSE: 42.5650
Holdout MSE: 46.4945
Holdout R2: 0.4409
Components used: 12


## (b)
```{admonition}
:class: note
Propose a model (or set of models) that seem to perform well on this data set, and justify your answer.

Each model seems to perform about the same. The model formed by subset selection or lasso may be preferred since they use less features and have reasonable test errors. Note that the current models were trained on a smaller subset, so we would prefer refitting on the entire data set for future predictions.

## (c)
```{admonition}
:class: note
Does your chosen model involve all of the features in the data set? Why or why not?

In [113]:
list(X.columns[best_lasso.named_steps['lasso'].coef_ != 0])

['dis', 'rad', 'lstat', 'medv']

The subset model chosen only includes 5 features. Lasso includes only 4 features. This can be preferabble to minimize variance due to variables that are mainly noise.